# Pipeline Orchestrator
**Apex Retail Intelligence — end-to-end run**

Chains all four pipeline notebooks in order using `dbutils.notebook.run`, so the whole
Bronze→Silver→Gold pipeline can be scheduled as a single Databricks Job (Workflow) with one
entry point. Each notebook remains independently runnable/testable on its own, as required.

**Suggested Job schedule:** daily, after the day's incremental CSV + audit files have been
uploaded to the inbound Volume.

In [0]:
dbutils.widgets.text("catalog_root", "/Volumes/apex_retail1/landing_zone/inbound_data", "Inbound Volume Root")
dbutils.widgets.text("pipeline_root", "/Volumes/apex_retail1/pipeline/data", "Pipeline Storage Root")

params = {
    "catalog_root": dbutils.widgets.get("catalog_root"),
    "pipeline_root": dbutils.widgets.get("pipeline_root"),
}

TIMEOUT_SECONDS = 3600

In [0]:
import time

steps = [
    ("01_raw_landing_ingestion", params),
    ("02_bronze_layer", {"pipeline_root": params["pipeline_root"]}),
    ("03_silver_layer", {"catalog_root": params["catalog_root"]}),
    ("04_gold_layer_kpi", {}),
]

run_log = []
for notebook, notebook_params in steps:
    start = time.time()
    print(f"▶ Running {notebook} ...")
    try:
        result = dbutils.notebook.run(notebook, TIMEOUT_SECONDS, notebook_params)
        elapsed = round(time.time() - start, 1)
        print(f"✅ {notebook} completed in {elapsed}s -> {result}")
        run_log.append((notebook, "SUCCESS", elapsed, str(result)))
    except Exception as e:
        elapsed = round(time.time() - start, 1)
        print(f"❌ {notebook} FAILED after {elapsed}s: {e}")
        run_log.append((notebook, "FAILED", elapsed, str(e)))
        raise  # stop the job — a downstream layer should never run on a failed upstream layer

▶ Running 01_raw_landing_ingestion ...
✅ 01_raw_landing_ingestion completed in 92.3s -> None
▶ Running 02_bronze_layer ...
✅ 02_bronze_layer completed in 42.7s -> None
▶ Running 03_silver_layer ...
✅ 03_silver_layer completed in 72.8s -> None
▶ Running 04_gold_layer_kpi ...
✅ 04_gold_layer_kpi completed in 62.0s -> None


In [0]:
display(spark.createDataFrame(run_log, ["notebook", "status", "elapsed_seconds", "detail"]))
print("\n🎉 Apex Retail Intelligence pipeline run complete: Raw → Landing → Bronze → Silver → Gold → KPIs.")

notebook,status,elapsed_seconds,detail
01_raw_landing_ingestion,SUCCESS,92.3,None
02_bronze_layer,SUCCESS,42.7,None
03_silver_layer,SUCCESS,72.8,None
04_gold_layer_kpi,SUCCESS,62.0,None



🎉 Apex Retail Intelligence pipeline run complete: Raw → Landing → Bronze → Silver → Gold → KPIs.
